# Kenya demographic reconstruction and prioritisation

This notebook uses the same functions and pipeline as the command-line scripts. It examines the source, cleaning, exploratory data analysis, missingness, grouped splits, masking, model selection, prediction-error costs, and simulated prioritisation. All results are exploratory decision support, not official estimates.

Run from top to bottom with the project environment and the local HDX CSV in `data/raw/`. The full training pipeline may take considerable time and creates a new run directory; it never overwrites a previous run. For an existing complete run with matching input and configuration checksums, set `AML_NOTEBOOK_RUN_DIR` before starting Jupyter. Only use trusted model artifacts.

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import display

root = Path.cwd().resolve()
if not (root / 'run_pipeline.py').exists():
    root = root.parent
if not (root / 'run_pipeline.py').exists():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks directory')
sys.path.insert(0, str(root))
from src.analysis_workflow import (  # noqa: E402
    evaluate_selected,
    exploratory_figures,
    exploratory_tables,
    model_figures,
    obtain_run,
    read_config,
    run_outputs,
    source_and_profiles,
    source_tables,
)

config_path = root / 'config.yaml'
config = read_config(config_path)
config

The configuration controls the fixed seed, raw input, output directory and simulation budget. Changing it changes the experiment; compare only runs with compatible inputs, splits and masks.

## Source and data understanding
The loader filters the local HDX HAPI file to Kenyan county records and checks that all 47 county codes exist. The field table describes the filtered records, not the global source.

In [ ]:
source, profiles = source_and_profiles(config)
source_info = source_tables(source)
display(source_info['source_overview'])
display(source_info['source_fields'])
display(source_info['source_categories'])
display(source.head())

Review the reference period, field completeness, category counts and duplicates above before drawing conclusions. Source values are initially read as strings; the cleaner validates and converts population values. A single historical extract cannot establish trends or causal relationships.

## Cleaning, distributions and missingness
Cleaning creates one county profile while preserving reported aggregate totals and missing observations. The same analysis functions also produce CSV and HTML reports in every new pipeline run.

In [ ]:
tables = exploratory_tables(profiles)
figures = exploratory_figures(tables)
display(tables['overview'])
display(profiles.head())
display(tables['county_totals'])
display(tables['atomic_distribution'])
display(tables['demographics'])
figures['county_population'].show()
figures['county_distribution'].show()
figures['age_sex'].show()
figures['dependency'].show()

The county chart uses reported totals; the age-sex table and chart summarize the 42 atomic cells. Large differences between counties or age groups can make absolute error and RMSE disproportionately sensitive to larger populations. These are counts, not rates adjusted for county population.

In [ ]:
display(tables['missing_by_county'])
display(tables['missing_by_column'])
display(tables['consistency_summary'])
display(tables['consistency_checks'].query("status != 'consistent'"))
figures['atomic_missingness'].show()

Missing values are not automatically zeros. Inconsistent source totals are reported rather than silently corrected; their differences also affect later consistency metrics. A complete source does not prove that simulated missingness matches real-world missingness.

## Reproducible pipeline: features, masks, training and saving
The shared pipeline creates disjoint county-grouped train, validation and test splits; simulates random, age-block and county-block masks; trains the tree and autoencoder candidates using grouped cross-validation; selects by validation MAE; saves model artifacts; and runs prioritisation on observed profiles. Test counties are not used in selection.

In [ ]:
existing = os.environ.get('AML_NOTEBOOK_RUN_DIR')
run_dir = obtain_run(config_path, Path(existing) if existing else None)
outputs = run_outputs(run_dir)
display({'run_directory': str(run_dir), 'manifest': outputs['manifest']})
display(outputs['experiments'])
display(outputs['selection'])

The manifest records checksums, seed and data-quality warnings. Each mask summary lists actual held-out cells, which may differ from the requested percentage. Selection averages the nine validation MAEs equally, and artifacts are in `models/selected` within this run. They are not automatically deployed. Reusing an existing run requires matching checksums, but these do not verify its original code revision or dependency versions.

## Model quality, generalisation and overfitting risk
Examine every scenario rather than only the average. MAE and RMSE are in persons per masked cell; RMSE emphasizes large errors. R-squared can be negative or undefined when targets are constant. Grouped CV uses training counties, while validation uses different counties.

In [ ]:
display(outputs['comparison'])
display(outputs['diagnostics'])
model_charts = model_figures(outputs)
model_charts['validation'].show()
model_charts['generalisation'].show()

The median baseline predicts each age-sex cell using only training-county medians. Compare its MAE and RMSE with the selected model for each scenario. A large validation-versus-CV gap warrants investigation, but these estimates use different small county samples and are not an in-sample training loss curve. County-block 10% has only three eligible training counties. Selection on seven validation counties adds optimism; no procedure here can guarantee absence of overfitting or good future performance. Consistency columns compare against potentially inconsistent reported totals. The autoencoder also receives aggregate training supervision unavailable to the trees.

## Cost of wrong predictions
Underestimation can lead to insufficient planned resources, while overestimation can lead to over-allocation. Their true costs depend on the decision and are unknown. The following sensitivity analysis assigns illustrative costs per person of (under, over) = (1, 1), (2, 1), and (1, 2).

In [ ]:
display(outputs['diagnostics'][['pattern', 'rate', 'targets', 'bias',
    'mean_underestimate', 'mean_overestimate', 'underestimate_fraction',
    'cost_equal', 'cost_under_double', 'cost_over_double']])
model_charts['costs'].show()

Costs are average hypothetical units per artificially masked cell, not currency, actual financial loss or a validated downstream decision outcome. Positive bias means overprediction. Equal-cost loss equals MAE; asymmetric losses increase when the corresponding error direction is more expensive. The MAE-selected model is not necessarily optimal for either asymmetric cost.

## Simulated prioritisation
The optimisation uses observed profiles rather than reconstructed predictions. Each selected county has one simulated cost unit, with child coverage, elderly coverage and a dependency-burden proxy as objectives.

In [ ]:
display(tables['indicators'])
display(outputs['optimisation'])
display(outputs['tradeoffs'])
model_charts['tradeoffs'].show()

Inspect trade-offs rather than treating a single solution as universally best. NSGA-III is approximate, and hypervolume depends on its reference point. The burden proxy is not validated fairness or actual service coverage; these simulated costs are not the prediction-error costs above.

## Optional final held-out test
Only after the methodology and selection are frozen, set `AML_EVALUATE_TEST=1` before starting the notebook to evaluate the selected family on the seven reserved test counties. Do not repeatedly tune decisions using these scores. The same evaluation is available through `python -m src.models.diagnostics --split test`.

In [ ]:
if os.environ.get('AML_EVALUATE_TEST') == '1':
    test_path = run_dir / 'reports' / 'selected_test_diagnostics.csv'
    if test_path.exists():
        raise FileExistsError(f'Test report already exists: {test_path}')
    test_results = evaluate_selected(run_dir / 'data' / 'kenya_county_population.csv',
        run_dir / 'models' / 'selected', split_name='test')
    test_results.to_csv(test_path, index=False)
    display(test_results)
else:
    print('Test evaluation skipped; held-out counties remain reserved.')

If enabled, this is a final estimate from a very small, simulated-mask test set, not proof of deployment readiness or protection against distribution shift. Preserve the report and disclose the sample size, uncertainty, source inconsistencies and all modeling assumptions.